In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

In [3]:
num_entities = 3
num_relations = 1
embedding_dim = 4

# Initialize the Lookup Tables with RANDOM numbers

In [4]:
entity_table = nn.Embedding(num_entities, embedding_dim)
relation_table = nn.Embedding(num_relations, embedding_dim)


print("Intial Entity Table")
print(entity_table.weight.data)

print("Intial Relation Table")
print(relation_table.weight.data)

Intial Entity Table
tensor([[-1.0827,  1.0851, -0.6536, -0.3613],
        [-0.4118, -0.1575,  0.0214, -0.8979],
        [ 0.1355,  2.0599,  1.6534, -0.3715]])
Intial Relation Table
tensor([[-1.9900, -0.0732, -0.4732,  0.3987]])


# Define a True Fact and a Fake Fact

In [5]:
head = torch.tensor([0])
rel  = torch.tensor([0])
tail_true = torch.tensor([1])
tail_fake = torch.tensor([2])

# Setup the Optimizer & Loss Function

In [6]:
optimizer = optim.SGD(list(entity_table.parameters()) + list(relation_table.parameters()), lr=0.1)
optimizer.zero_grad() # Clear any old memory

# Loss Function (Margin based)
margin_loss = nn.MarginRankingLoss(margin=2.0)

In [7]:
head_vec = entity_table(head)
relation_vec = relation_table(rel)
tail_true_vec = entity_table(tail_true)
tail_fake_vec = entity_table(tail_fake)

head_vec

tensor([[-1.0827,  1.0851, -0.6536, -0.3613]], grad_fn=<EmbeddingBackward0>)

# Scoring Function (Distance calculation)

In [102]:
dist_true = torch.norm( head_vec + relation_vec - tail_true_vec, p=1 ).unsqueeze(0)
dist_fake = torch.norm( head_vec + relation_vec - tail_fake_vec, p=1 ).unsqueeze(0)

print("dist_true", dist_true)
print("dist_fake",dist_fake)

dist_true tensor([5.5670], grad_fn=<UnsqueezeBackward0>)
dist_fake tensor([7.4253], grad_fn=<UnsqueezeBackward0>)


# Loss Function compares true vs fake distance

In [103]:
# target=-1 tells PyTorch that dist_fake should be LARGER than dist_true
target = torch.tensor([-1.0])
loss = margin_loss(dist_true, dist_fake, target)
print("loss = ",loss)

loss =  tensor(0.1417, grad_fn=<MeanBackward0>)


# Backpropagation & Optimizer edit

In [104]:
loss.backward()
optimizer.step() # Optimizer directly edit lookup Table

In [105]:
print("Entity Table after Backpropogation & Optimizer edit")
print(entity_table.weight.data)

Entity Table after Backpropogation & Optimizer edit
tensor([[-0.2916, -0.2421, -2.4453,  0.8888],
        [ 1.0525,  0.3448, -1.5555,  0.3242],
        [-0.8748,  2.3374, -0.4967,  1.8585]])
